In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
cwd = os.getcwd()
if os.path.basename(cwd) == 'notes':
    os.chdir('..')
print('当前工作目录:', os.getcwd())

In [ ]:
# ============================================================
# XGBoost 时序预测 (23 通道: 气象+三角编码+独热+rolling)
# 官方文档 API: DMatrix → xgb.train → bst.predict
# ============================================================

# --- 1. 读取原始数据 ---
df_train_raw = pd.read_csv('data/ShanghaiPM_Training.csv', na_values='NA')
df_test_raw  = pd.read_csv('data/ShanghaiPM_Test.csv',  na_values='NA')

# --- 2. 数据清洗 + 特征工程 ---
def preprocess(df):
    """预处理: 时间戳 → pm_ave(log) → 三角编码 → 独热 → rolling"""
    df['datetime'] = pd.to_datetime(
        {'year': df['year'], 'month': df['month'], 'day': df['day'], 'hour': df['hour']})
    df = df.set_index('datetime').sort_index()

    # pm_ave = 三站均值, 删除全缺失行
    pm_cols = ['PM_Jingan', 'PM_US Post', 'PM_Xuhui']
    df['pm_ave'] = df[pm_cols].mean(axis=1)
    df = df.dropna(subset=['pm_ave'])

    # log(1+x) 变换: 压缩高值, 防止模型过度拟合极端 PM2.5
    df['pm_ave'] = np.log1p(df['pm_ave'])

    # --- 周期变量: sin+cos 成对编码 ---
    hour = df.index.hour.values.astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)

    month = df.index.month.values.astype(float)
    df['month_sin'] = np.sin(2 * np.pi * month / 12)
    df['month_cos'] = np.cos(2 * np.pi * month / 12)

    dow = df.index.dayofweek.values.astype(float)
    df['dayofweek_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * dow / 7)

    # --- 风向 cbwd: One-Hot 独热编码 (5种风向 → 5个独立 0/1 通道) ---
    cbwd_categories = ['cv', 'SE', 'NW', 'SW', 'NE']
    df['cbwd'] = pd.Categorical(df['cbwd'], categories=cbwd_categories)
    cbwd_dummies = pd.get_dummies(df['cbwd'], prefix='cbwd').astype(float)
    df = pd.concat([df, cbwd_dummies], axis=1)
    df = df.drop('cbwd', axis=1)

    # --- Rolling 统计量 (过去 6h 和 24h 的 pm_ave 均值/标准差) ---
    # 在 log 空间计算, min_periods=1 保证首行也有值
    df['pm_roll6_mean']  = df['pm_ave'].rolling(6,  min_periods=1).mean()
    df['pm_roll6_std']   = df['pm_ave'].rolling(6,  min_periods=1).std().fillna(0)
    df['pm_roll24_mean'] = df['pm_ave'].rolling(24, min_periods=1).mean()
    df['pm_roll24_std']  = df['pm_ave'].rolling(24, min_periods=1).std().fillna(0)

    return df

df_train = preprocess(df_train_raw)
df_test  = preprocess(df_test_raw)
print(f'训练集: {df_train.shape}  |  测试集: {df_test.shape}')

# --- 3. 构建特征矩阵 ---
# 23 通道 × 24 步 = 552 维:
#   pm_ave(log), DEWP, HUMI, PRES, TEMP, Iws, precipitation, Iprec,  (8)
#   hour_sin, hour_cos, month_sin, month_cos,                        (4)
#   dayofweek_sin, dayofweek_cos,                                    (2)
#   cbwd_cv, cbwd_SE, cbwd_NW, cbwd_SW, cbwd_NE,                     (5)
#   pm_roll6_mean, pm_roll6_std, pm_roll24_mean, pm_roll24_std       (4)
FEATURE_COLS = [
    'pm_ave', 'DEWP', 'HUMI', 'PRES', 'TEMP', 'Iws', 'precipitation', 'Iprec',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'cbwd_cv', 'cbwd_SE', 'cbwd_NW', 'cbwd_SW', 'cbwd_NE',
    'pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std',
]
tau = 24
n_features = len(FEATURE_COLS)    # 23

# rolling 特征的索引 (预测时需动态重算, 防止偷看未来真实值)
ROLLING_COLS = ['pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std']
rolling_indices = [FEATURE_COLS.index(c) for c in ROLLING_COLS]
pm_idx = FEATURE_COLS.index('pm_ave')

def build_features(df, feature_cols, tau=24):
    """构建滑窗特征矩阵: 每行 = 过去 tau 步 × n_features 通道"""
    n = len(df)
    data = df[feature_cols].values
    X = np.zeros((n - tau, tau * len(feature_cols)))
    for t in range(tau):
        X[:, t * len(feature_cols):(t + 1) * len(feature_cols)] = data[t: n - tau + t]
    y = df['pm_ave'].values[tau:]
    return X, y

# 从 df_train 切出最后 20% 作为验证集 (按时间顺序)
split_idx = int(len(df_train) * 0.8)
df_val = df_train.iloc[split_idx:]
df_train = df_train.iloc[:split_idx]
print(f'训练集: {len(df_train)}  |  验证集: {len(df_val)}  |  测试集: {len(df_test)} (训练时不碰)')

X_train, y_train = build_features(df_train, FEATURE_COLS, tau)
X_val,   y_val   = build_features(df_val,   FEATURE_COLS, tau)
print(f'训练特征: {X_train.shape}  |  验证特征: {X_val.shape}  |  每行 {tau * n_features} 维')

# --- 4. 创建 DMatrix (官方数据结构) ---
feature_names = [f'{col}_t-{tau - t}' for t in range(tau) for col in FEATURE_COLS]
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=feature_names)

# --- 5. 设置参数 (官方文档写法, 加大过拟合惩罚) ---
params = {
    'max_depth': 5,                    # 树深度 (限制复杂度)
    'eta': 0.03,                       # 学习率 (保守)
    'objective': 'reg:squarederror',   # 回归目标
    'eval_metric': 'rmse',             # 评估指标
    'tree_method': 'hist',             # 直方图算法 (官方推荐)
    'subsample': 0.7,                  # 行采样
    'colsample_bytree': 0.4,           # 列采样 (552维需更激进)
    'min_child_weight': 10,            # 叶节点最小权重
    'gamma': 0.2,                      # 分裂最小增益
    'lambda': 2.0,                     # L2 正则化
    'alpha': 0.5,                      # L1 正则化
    'nthread': 4,
}

evals = [(dtrain, 'train'), (dval, 'eval')]

# --- 6. 训练 (带早停, 官方文档推荐) ---
bst = xgb.train(
    params, dtrain,
    num_boost_round=1500,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=20,
)
print(f'\n最佳轮次: {bst.best_iteration}  |  最佳 RMSE (log空间): {bst.best_score:.4f}')

# --- 7. 特征重要性 (官方: xgb.plot_importance) ---
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(bst, max_num_features=30, ax=ax)
plt.title('Feature Importance (Top 30)')
plt.tight_layout()
plt.show()

print(f'\n模型训练完成, 进入 20 组评估...')

In [ ]:
# ============================================================
# 20 组评估: 每组 24h 上下文 + 72h 预测
# pm_ave 在 log 空间训练/预测, 最后用 expm1 还原
# rolling 统计量在预测时动态重算 (防止偷看未来真实值)
# 评估指标: RMSE
# ============================================================
N_GROUPS = 20
N_CONTEXT = 24
N_PREDICT = 72
STRIDE = (len(df_test) - N_CONTEXT - N_PREDICT) // (N_GROUPS - 1)

test_data = df_test[FEATURE_COLS].values
pm_test = df_test['pm_ave'].values  # log 空间

all_rmses = []
fig, axes = plt.subplots(5, 4, figsize=(20, 20))
axes = axes.flatten()

for g in range(N_GROUPS):
    start = g * STRIDE
    # pm_series: log 空间的 96 小时 pm_ave (前 24 为上下文真实值)
    pm_series = np.zeros(N_CONTEXT + N_PREDICT)
    pm_series[:N_CONTEXT] = pm_test[start:start + N_CONTEXT]

    # 多步预测 72 小时 (在 log 空间进行)
    for i in range(N_PREDICT):
        window = np.zeros(tau * n_features)
        for t in range(tau):
            local_pos = i + t          # 在 pm_series 中的位置
            abs_idx = start + i + t    # 在 df_test 中的绝对位置
            base = t * n_features
            # pm_ave 通道
            window[base + pm_idx] = pm_series[local_pos]
            # 其他通道
            for ch in range(n_features):
                if ch == pm_idx:
                    continue
                elif ch in rolling_indices:
                    # rolling 统计量: 预测区间用 pm_series 动态重算 (防作弊)
                    if local_pos >= N_CONTEXT:
                        col_name = FEATURE_COLS[ch]
                        w = 6 if 'roll6' in col_name else 24
                        s = max(0, local_pos - w + 1)
                        slice_ = pm_series[s:local_pos + 1]
                        if 'mean' in col_name:
                            window[base + ch] = np.mean(slice_)
                        else:
                            window[base + ch] = np.std(slice_) if len(slice_) > 1 else 0.0
                    else:
                        window[base + ch] = test_data[abs_idx, ch]
                else:
                    # 外生变量: 始终用真实值
                    window[base + ch] = test_data[abs_idx, ch]
        dpredict = xgb.DMatrix(window.reshape(1, -1), feature_names=feature_names)
        pm_series[N_CONTEXT + i] = bst.predict(
            dpredict, iteration_range=(0, bst.best_iteration + 1))[0]

    # expm1 还原到原始量纲 (μg/m³)
    preds_log = pm_series[N_CONTEXT:]
    actual_log = pm_test[start + N_CONTEXT:start + N_CONTEXT + N_PREDICT]
    preds  = np.expm1(preds_log)
    actual = np.expm1(actual_log)

    # RMSE
    rmse = np.sqrt(((preds - actual) ** 2).mean())
    all_rmses.append(rmse)

    ax = axes[g]
    hours = np.arange(1, N_PREDICT + 1)
    ax.plot(hours, actual, linewidth=0.8, label='actual')
    ax.plot(hours, preds, linewidth=0.8, label='predicted')
    ax.set_title(f'Group {g+1} (start={start}), RMSE={rmse:.1f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('hour')
    ax.set_ylabel('PM2.5 (μg/m³)')

plt.suptitle('XGBoost — 20 Groups (24h context → 72h prediction)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f'\n=== 20 组 RMSE 汇总 ===')
for g, rmse in enumerate(all_rmses):
    print(f'  Group {g+1:2d}: RMSE = {rmse:.2f}')
print(f'\n平均 RMSE = {np.mean(all_rmses):.2f}')